**Summons Applicable libraries**

Summons libraries for hashing, time-stamps, and randomisation

In [25]:
import hashlib
import time
import random

**Tic-Tac-Toe Game Engine**

Defines a class that facilitates the game of Tic-Tac-Toe

Has functions for initialisation, displaying board, executing moves, switching current players (to ensure each player gets only one turn at a time), checking wins, ending games that result in draws, getting available moves, and randomising moves (see discussion in my report)

In [26]:
class TicTacToe:

  #Initialisation function
  def __init__(self):
    # 3x3 board initialised with empty spaces
    self.board = [" " for _ in range(9)]
    self.current_player = "X"  # Alice starts

  #Display function
  def display_board(self):
    for i in range(0, 9, 3):
      print(self.board[i], "|", self.board[i+1], "|", self.board[i+2])

  #Executes moves
  def make_move(self, position):
    # position: 0–8
    if self.board[position] != " ":
      print("Invalid move! Spot already taken.")
      return False

    self.board[position] = self.current_player
    return True

  #Switches to let other player have turn - important in ensuring each player gets their rightful turn
  def switch_player(self):
    self.current_player = "O" if self.current_player == "X" else "X"

  #Checks for winners
  def check_winner(self):
    combos = [
        (0,1,2),(3,4,5),(6,7,8),
        (0,3,6),(1,4,7),(2,5,8),
        (0,4,8),(2,4,6)
    ]

    for combo in combos:
        a,b,c = combo
        #print(f"Checking {combo}: {self.board[a]}, {self.board[b]}, {self.board[c]}")  # DEBUG

        if self.board[a] == self.board[b] == self.board[c] != " ":
            return self.board[a], combo

    return None, None

  #Determines if there are any draws or not
  def is_draw(self):
    return " " not in self.board

#Identifies if any available moves
def get_available_moves(board):
  return [i for i in range(9) if board[i] == " "]

#Randomises moves
def choose_random_move(board):
  available = get_available_moves(board)
  return random.choice(available)

**Applies Cryptographic Features**

That means hashes and timestamps

In [27]:
def commit_state(board):
    timestamp = str(time.time()) #Assigns a timestamp
    nonce = str(random.random()) #Assigns a nonce

    data = str(board) + timestamp + nonce #Adds board + timestamp + nonce
    commitment = hashlib.sha256(data.encode()).hexdigest() #Hashes the previous sum

    #Returns the hash, timestamp, and nonce
    return {
        "commitment": commitment,
        "timestamp": timestamp,
        "nonce": nonce#, #This will be returned now, but will not be provided to Bob/verifiers
        #"board": board.copy()
    }

**Facilitates Gameplay**

Uses previously defined functions in game engine to achieve said tasks.

In [28]:
def play_automated_game():
  game = TicTacToe()
  private_log = []
  public_log = []

  #Keeps the game running through randomisation until a results are made
  while True:
      move = choose_random_move(game.board) #Randomisation (see discussion)

      #Makes the move
      #print(f"Player {game.current_player} plays {move}")
      game.make_move(move)

      #Log move
      entry = commit_state(game.board) #Hash it

      #Adds to private log (Available to Alice only) - not used in zk-SNARKs
      private_log.append({
          "player": game.current_player,
          "move": move,
          "board": game.board.copy(),
          "timestamp": entry["timestamp"],
          "nonce": entry["nonce"]
      })

      #Adds to public log (Bob sees this and used by zk-SNARKs)
      public_log.append({
          "commitment": entry["commitment"],
          "timestamp": entry["timestamp"]
      })

      winner, combo = game.check_winner()
      if winner:
          print("--- CLAIM (Win Detected) ---")  # DEBUG
          game.display_board()
          print(f"Alice (X) claims she won by securing: {combo}")
          #if winner == "X":
              #print(f"Alice (X) claims she won and we agree with her! Line: {combo}")
          #elif winner == "O":
              #print(f"Alice (X) claims she won. We think her opponent (O) wins. Line: {combo}")
          return game, private_log, public_log, winner, combo

      if game.is_draw():
          #game.display_board()
          #print("Draw!")
          return game, private_log, private_log, None, None

      game.switch_player()

**Verifier Function**

Gets the proof and verifies it

In [29]:
def verify_public_proof(public_log, final_board):
    print("\n--- Verifying Zero-Knowledge Proof ---")

    #Timestamp check - ensures no forgery
    for i in range(1, len(public_log)):
        if float(public_log[i]["timestamp"]) < float(public_log[i-1]["timestamp"]):
            print("Timestamp order invalid")
            return False

    #Compute actual result
    game = TicTacToe()
    game.board = final_board

    real_winner, real_combo = game.check_winner() #Identifies real winner after checking the board - ONLY CURRENT STATE

    #Announces actual result
    if real_winner == "X":
        print("Verified: Alice (X) wins.")
        return #True

    elif real_winner == "O":
        print("Verification failed: Alice lost (O wins)")
        return #False

    else:
        print("raw: Alice did not win")
        return #False

**Main Function**

Executes gameplay and the zk-SNARKs proof is done in proper order

In [30]:
#def execute():
#print("Running automated Tic-Tac-Toe...\n")

while True:
        game, private_log, public_log, winner, combo = play_automated_game() #Executes gameplay to generate

        if winner is not None:  # only accept winning games
            break  # only accept winning games

#Summons proof algorithm
print("\n--- PROOF ---")
for step in public_log:
    print(step)

#Summons verification
verify_public_proof(public_log, game.board#, winner, combo
                    )

#execute()

--- CLAIM (Win Detected) ---
X |   |  
O | X | X
O | O | X
Alice (X) claims she won by securing: (0, 4, 8)

--- PROOF ---
{'commitment': '11155bb03fcbd3952ce8cd5c7f6df1553cc271a807cda851918e8827fc8d299f', 'timestamp': '1775349940.4112327'}
{'commitment': '8a73ddd9ef77dfaa08e64e1b96a4c503b91b20012e94c58ca8f9fce2e3fefc87', 'timestamp': '1775349940.411296'}
{'commitment': '56f475ca861b8706fc30f55a7c13d0896fcb65a922612126f61299bf2bf416ee', 'timestamp': '1775349940.4113233'}
{'commitment': '077225be2b00babafd9f7c147c280978d9eac18c94e2f9f8b52606bcb8c6db77', 'timestamp': '1775349940.4113567'}
{'commitment': '1421a604eb8a5b93bef2159e20fe3ef225017c190528064749e9d195bfa30df7', 'timestamp': '1775349940.4113839'}
{'commitment': '20900e146d07365000ac544a517b0bd6a6f44699a8e758f90af6ee15199ba60a', 'timestamp': '1775349940.4114006'}
{'commitment': 'de70a07973b1795d1ea23d8db536bd8da8644433ea42f887f5aedb2a1adaf05f', 'timestamp': '1775349940.411416'}

--- Verifying Zero-Knowledge Proof ---
Verified: Alic